# Validating Namsor classifications + creating final gender classifications for Australia and Canada

1. load in namsor classifications and metadata-based gender classifications
2. validation process of Namsor using ground-truth names
3. create final gender classification
4. save final classification to files

## 1. Load in different gender classifications

In [1]:
# import libraries
import pickle
import statistics
import matplotlib.pyplot as plt
import re
import pandas as pd
import numpy as np
from collections import Counter
from collections import OrderedDict
from collections import defaultdict
import math

In [2]:
# load files

# Namsor
with open('../data/canada/author_analysis/namsor_can_final_29jan2026.pkl', 'rb') as f:
    namsor_can = pickle.load(f, encoding="utf-8")
    
with open('../data/aus/author_analysis/namsor_aus_final_29jan2026.pkl', 'rb') as f:
    namsor_aus = pickle.load(f, encoding="utf-8")
    
    
# gender_guesser
with open('../data/canada/author_analysis/can_gender_guesser.pkl', 'rb') as f:
    gen_guesser_can = pickle.load(f, encoding="utf-8")
    
with open('../data/aus/author_analysis/aus_gender_guesser.pkl', 'rb') as f:
    gen_guesser_aus = pickle.load(f, encoding="utf-8")
    

# metadata
with open('../data/canada/author_analysis/can_metadata_only_gen.pkl', 'rb') as f:
    metadata_can = pickle.load(f, encoding="utf-8")
    
with open('../data/aus/author_analysis/aus_metadata_only_gen.pkl', 'rb') as f:
    metadata_aus = pickle.load(f, encoding="utf-8")

In [3]:
# Load in dictionary of Australian IDs for validating Namsor gender process

with open('../data/aus/author_analysis/aus_namsor_validation_ids.pkl', 'rb') as f:
    aus_validation_ids = pickle.load(f, encoding="utf-8")

In [4]:
#aus_validation_ids

## 2. Validate Namsor and gender_guesser classifications using the set of Australian validation IDs

In [5]:
# full list of IDs
validation_ids_full = aus_validation_ids['in'] + aus_validation_ids['out']

print(len(validation_ids_full))
print(len(set(validation_ids_full)))


print(len(aus_validation_ids['in']))
print(len(set(aus_validation_ids['in'])))

print(len(aus_validation_ids['out']))
print(len(set(aus_validation_ids['out'])))

# IDs of names in canada baby name collections

# IDs of names out of canada baby name collections

2614
2614
2291
2291
323
323


In [6]:
def check_ag(d1, d2, key_list):
    
    ag = 0
    non_ag = 0    
    for k in key_list:
        if d1[k] == d2[k]:
            ag += 1
        else:
            non_ag += 1
            
    if ag + non_ag != len(key_list):
        print('Warning: number of agree/non-agree checks != number of keys in list')
        
    result_dict = {}
    result_dict['ag'] = ag
    result_dict['non_ag'] = non_ag
    result_dict['ag_prop'] = ag/len(key_list)
    result_dict['non_ag_prop'] = non_ag/len(key_list)
        
    return result_dict

### 2.1 Namsor accuracy rate

In [7]:
acc = check_ag(namsor_aus, metadata_aus, validation_ids_full)

print('Namsor accuracy - all validation petitions')
print(acc['ag'])
print(acc['non_ag'])
print(round(acc['ag_prop']*100, 2))
print(round(acc['non_ag_prop']*100, 2))

Namsor accuracy - all validation petitions
2432
182
93.04
6.96


In [8]:
acc = check_ag(namsor_aus, metadata_aus, aus_validation_ids['in'])

print("Namsor accuracy - Validation petitions where the author's first name was in the Canadian baby names")
print(acc['ag'])
print(acc['non_ag'])
print(round(acc['ag_prop']*100, 2))
print(round(acc['non_ag_prop']*100, 2))

Namsor accuracy - Validation petitions where the author's first name was in the Canadian baby names
2205
86
96.25
3.75


In [9]:
acc = check_ag(namsor_aus, metadata_aus, aus_validation_ids['out'])

print("Namsor accuracy - Validation petitions where the author's first name was NOT in the Canadian baby names")
print(acc['ag'])
print(acc['non_ag'])
print(round(acc['ag_prop']*100, 2))
print(round(acc['non_ag_prop']*100, 2))

Namsor accuracy - Validation petitions where the author's first name was NOT in the Canadian baby names
227
96
70.28
29.72


### 2.2 gender_guesser accuracy rate

In [10]:
acc = check_ag(gen_guesser_aus, metadata_aus, validation_ids_full)

print('gender_guesser accuracy - all validation petitions')
print(acc['ag'])
print(acc['non_ag'])
print(round(acc['ag_prop']*100, 2))
print(round(acc['non_ag_prop']*100, 2))

gender_guesser accuracy - all validation petitions
2167
447
82.9
17.1


In [11]:
acc = check_ag(gen_guesser_aus, metadata_aus, aus_validation_ids['in'])

print("gender_guesser accuracy - Validation petitions where the author's first name was in the Canadian baby names")
print(acc['ag'])
print(acc['non_ag'])
print(round(acc['ag_prop']*100, 2))
print(round(acc['non_ag_prop']*100, 2))

gender_guesser accuracy - Validation petitions where the author's first name was in the Canadian baby names
2089
202
91.18
8.82


In [12]:
acc = check_ag(gen_guesser_aus, metadata_aus, aus_validation_ids['out'])

print("gender_guesser accuracy - Validation petitions where the author's first name was NOT in the Canadian baby names")
print(acc['ag'])
print(acc['non_ag'])
print(round(acc['ag_prop']*100, 2))
print(round(acc['non_ag_prop']*100, 2))

gender_guesser accuracy - Validation petitions where the author's first name was NOT in the Canadian baby names
78
245
24.15
75.85


### 2.3 Namsor error types

In [24]:
def check_errors(df1, df2, id_list):
    c_o = 0
    c_f = 0
    c_m = 0
    non_ag_counter = 0
    unknown_error = 0
    misgendering_error = 0
    for k in id_list:
        if df1[k] != df2[k]:
            
            non_ag_counter += 1
            
            if df1[k] == 'unknown/other': # df2 should be the 'ground-truth' dataset
                c_o += 1
            elif df1[k] == 'female':
                c_f += 1
            else:
                c_m += 1
            
#             if df1[k] == 'unknown/other' and df2[k] != 'unknown/other': # unknown classification, true is male/female
#                 unknown_error += 1
#             elif df1[k] != 'unknown/other' and df2[k] != 'unknown/other': # both not male/female - swapped genders
#                 misgendering_error += 1
                
            if (df1[k] == 'unknown/other' and df2[k] in ['male', 'female']):
                unknown_error += 1
            elif ((df1[k] == 'male' and df2[k] == 'female') or (df1[k] == 'female' and df2[k] == 'male')):
                misgendering_error += 1
        
    errors = {'unknown/other': c_o, 'female': c_f, 'male': c_m, 
            'prop_misgender_total': misgendering_error/len(id_list), 
            'prop_unknown_total': unknown_error/len(id_list), 
            'prop_misgender_error': misgendering_error/non_ag_counter, 
            'prop_unknown_error': unknown_error/non_ag_counter,
           'total_errors': non_ag_counter}
    
    print()
    print(f"Number of misgendering errors: {misgendering_error}")
    print(f"Proportion of all errors that are misgendering errors: {round(errors['prop_misgender_error']*100, 2)}")
    
    print()
    print(f"Number of unknown errors: {unknown_error}")
    print(f"Proportion of all errors that are unknown errors: {round(errors['prop_unknown_error']*100, 2)}")
    
    print()
    print(f"Total errors: {errors['total_errors']}")
    print("*****\n")
    
    return errors


    

In [25]:
print('Namsor errors - all validation petitions')
errors = check_errors(namsor_aus, metadata_aus, validation_ids_full)

print("Namsor errors - Validation petitions where the author's first name was in the Canadian baby names")
errors = check_errors(namsor_aus, metadata_aus, aus_validation_ids['in'])

print("Namsor errors - Validation petitions where the author's first name was NOT in the Canadian baby names")
errors = check_errors(namsor_aus, metadata_aus, aus_validation_ids['out'])

Namsor errors - all validation petitions

Number of misgendering errors: 59
Proportion of all errors that are misgendering errors: 32.42

Number of unknown errors: 116
Proportion of all errors that are unknown errors: 63.74

Total errors: 182
*****

Namsor errors - Validation petitions where the author's first name was in the Canadian baby names

Number of misgendering errors: 33
Proportion of all errors that are misgendering errors: 38.37

Number of unknown errors: 47
Proportion of all errors that are unknown errors: 54.65

Total errors: 86
*****

Namsor errors - Validation petitions where the author's first name was NOT in the Canadian baby names

Number of misgendering errors: 26
Proportion of all errors that are misgendering errors: 27.08

Number of unknown errors: 69
Proportion of all errors that are unknown errors: 71.88

Total errors: 96
*****



### 2.4 gender_guesser error types

In [26]:
print('gender_guesser errors - all validation petitions')
errors = check_errors(gen_guesser_aus, metadata_aus, validation_ids_full)

print("gender_guesser errors - Validation petitions where the author's first name was in the Canadian baby names")
errors = check_errors(gen_guesser_aus, metadata_aus, aus_validation_ids['in'])

print("gender_guesser errors - Validation petitions where the author's first name was NOT in the Canadian baby names")
errors = check_errors(gen_guesser_aus, metadata_aus, aus_validation_ids['out'])


gender_guesser errors - all validation petitions

Number of misgendering errors: 66
Proportion of all errors that are misgendering errors: 14.77

Number of unknown errors: 376
Proportion of all errors that are unknown errors: 84.12

Total errors: 447
*****

gender_guesser errors - Validation petitions where the author's first name was in the Canadian baby names

Number of misgendering errors: 51
Proportion of all errors that are misgendering errors: 25.25

Number of unknown errors: 146
Proportion of all errors that are unknown errors: 72.28

Total errors: 202
*****

gender_guesser errors - Validation petitions where the author's first name was NOT in the Canadian baby names

Number of misgendering errors: 15
Proportion of all errors that are misgendering errors: 6.12

Number of unknown errors: 230
Proportion of all errors that are unknown errors: 93.88

Total errors: 245
*****



### 2.5 Concordance between Namsor and Canada metadata

In [20]:
c_total = 0
c_ag = 0


for k in metadata_can.keys():
    if metadata_can[k] != None:
        c_total += 1
        if metadata_can[k] == namsor_can[k]:
            c_ag +=1
        else:
            print(metadata_can[k])
            print(namsor_can[k])
            print("***")
            
print(c_total)
print(c_ag)
print(c_ag/c_total)

female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
male
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
male
unknown/other
***
female
unknown/other
***
female
unknown/other
***
male
unknown/other
***
female
unknown/other
***
female
unknown/other
***
male
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
male
unknown/other
***
female
unknown/other
***
male
unknown/other
***
female
unknown/other
***
female
unknown/other
***
male
female
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
female
unknown/other
***
1836
1797
0.9787581699346405


## 3. Create final mixed-method gender classifications

In [21]:
# Canada final dictionary
can_final_classification = {}

for k in metadata_can.keys():
    if metadata_can[k] != None:
        can_final_classification[k] = metadata_can[k]
    else:
        can_final_classification[k] = namsor_can[k]


In [22]:
# check
for k in can_final_classification.keys():
    if metadata_can[k] == None and can_final_classification[k] != namsor_can[k]:
        print("Non-match")
    if metadata_can[k] != None and can_final_classification[k] != metadata_can[k]:
        print("Non-match")

In [26]:
print(len([x for x in can_final_classification.values() if x == 'male']))
print(len([x for x in can_final_classification.values() if x == 'female']))
print(len([x for x in can_final_classification.values() if x == 'unknown/other']))

1061
932
41


In [23]:
# Australia final dictionary
aus_final_classification = {}

for k in metadata_aus.keys():
    if metadata_aus[k] != None:
        aus_final_classification[k] = metadata_aus[k]
    else:
        aus_final_classification[k] = namsor_aus[k]

In [24]:
# check
for k in aus_final_classification.keys():
    if metadata_aus[k] == None and aus_final_classification[k] != namsor_aus[k]:
        print("Non-match")
    if metadata_aus[k] != None and aus_final_classification[k] != metadata_aus[k]:
        print("Non-match")

In [27]:
print(len([x for x in aus_final_classification.values() if x == 'male']))
print(len([x for x in aus_final_classification.values() if x == 'female']))
print(len([x for x in aus_final_classification.values() if x == 'unknown/other']))

2366
1514
26


In [25]:
# write to files - uncomment if running for the first time

# with open('../data/canada/author_analysis/can_final_gen_classification.pkl', 'wb') as f:
#     pickle.dump(can_final_classification, f)
    
# with open('../data/aus/author_analysis/aus_final_gen_classification.pkl', 'wb') as f:
#     pickle.dump(aus_final_classification, f)